# Teach a little droid to walk
**163A / 263A · Robot kinematics and reinforcement learning**

Build a simulation, inspect the robot's joints, train a walking controller,
then change one reward and see how its gait changes.

**Today's questions:** What does the robot observe? What does its action mean?
How does a reward influence movement? How do we check whether it learned?

**Route through the lesson:** robot → environment → kinematics → control problem
→ SAC → your walking policy → one controlled experiment → optional hopping → play.

Examples are part of this text: images and short, silent looping clips. Run the exercises
to create your own results. Timings are from a local L40S; Colab remains untested.

## 1 · Meet the droid

The real-world demonstration is Disney's BDX droid. Our classroom uses the
separately developed **BDX-R open-source model**, simulated with **MuJoCo Warp**
through mjlab. We train with **FastSAC**; this is not Disney's controller.


| Real-world inspiration | Our simulated walker |
|:---:|:---:|
| ![Disney BDX droid — D23 reference photograph](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-disney_bdx.png?v=572a2471b677) | ![Recorded instructor walking example](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/inline/walking.gif?v=bf4f3466787a) |

Head gestures are cosmetic animation; leg motion comes from simulation.

Photo: [D23 / Disney](https://d23.com/bdx-droids-disneys-perfect-marriage-of-creativity-and-technological-achievement/), not relicensed. Simulation: the open-source BDX-R model with our recorded walking policy.

**Before running anything:** which joints do you expect to be most important
for moving a foot forward? Which might help keep the body upright?

Credits: [BDX-R model](https://github.com/BDX-R/BDX-R-MjLab),
[FastSAC project](https://younggyo.me/fastsac-humanoid/),
[Holosoma](https://github.com/amazon-far/holosoma), and
[mjlab](https://github.com/mujocolab/mjlab). Pinned revisions and license notices
are included with the lesson files.

## 2 · Create the environment

Select **Runtime → Change runtime type → GPU**. Installation may take several
minutes; it is separate from training time.

An environment combines the robot, ground, actuators, observations, reward,
and rules for ending an episode. We first inspect **one** robot. Training will
use many independent copies of this same task.

### Instructor example · the environment you will build

| The model at reset |
|:---:|
| ![BDX-R at its initial pose](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-robot.png?v=9e529632e115) |

This is a render of the actual model at reset, before learning.

| Setting | Baseline | Meaning |
|---|---|---|
| Physics update | 5 ms | Integrate forces and contacts |
| Policy update | 20 ms | Choose new joint targets |
| Episode limit | 10 s | Reset and try again |
| Speed command | −0.25 to +0.50 m/s | Backward through fast forward |
| Turn command | −0.60 to +0.60 rad/s | Right and left turns |

### Your activity · install, configure, and render

Run the setup and environment cells below. Your own model render will appear
beneath the final display cell. Compare its pose with the reference above.

In [ ]:
# @title Unpack the lesson files
import json
import os
from pathlib import Path
import subprocess
import sys
import time
from IPython.display import display, HTML, Image, Video

# The release is pinned so saved notebook copies keep using the same lesson code.
BUNDLE_URL = "https://github.com/TheNewRobot/bdxr-fastsac-classroom/releases/download/lesson-cb7f231815e9/BDXR_lesson_files.zip"
BUNDLE_SHA256 = "cb7f231815e92a8e8b9abb29da9e2e7ff22332cf4ac493fa9bed10c18ee892e6"
WORK = (
    Path("/content/bdxr_classroom")
    if Path("/content").exists()
    else Path.cwd() / "bdxr_classroom_lab"
) / BUNDLE_SHA256[:12]

"""Standard-library bootstrap, included verbatim in the notebook's setup cell.

Download one immutable release archive without querying Colab's browser. Files
are verified and staged before installation; student outputs are never removed.
"""

import hashlib
import json
import shutil
import tempfile
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path


def install_lesson(url, sha256, work, timeout=120, attempts=2):
    """Install a verified archive, with bounded requests and visible progress."""
    if timeout <= 0 or attempts < 1:
        raise ValueError("Download timeout and attempts must be positive.")
    work = Path(work)
    marker = work / ".lesson_release.json"
    if marker.is_file():
        try:
            installed = json.loads(marker.read_text())
        except (ValueError, OSError):
            installed = {}
        if installed.get("sha256") not in (None, sha256):
            raise RuntimeError(
                "This folder contains a different lesson release. "
                "Choose a new WORK folder to preserve your existing experiments."
            )
        if installed.get("files") and all(
            (work / name).is_file() for name in installed["files"]
        ):
            print(f"Lesson files already ready: {work}", flush=True)
            return
    work.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".bdxr-download-", dir=work.parent) as tmp:
        archive = Path(tmp) / "lesson.zip"
        for attempt in range(1, attempts + 1):
            start = time.monotonic()
            print(
                f"Downloading lesson files (attempt {attempt}/{attempts})…", flush=True
            )
            digest = hashlib.sha256()
            try:
                request = urllib.request.Request(
                    url, headers={"User-Agent": "BDXR-Lesson"}
                )
                with urllib.request.urlopen(
                    request, timeout=min(20, timeout)
                ) as response:
                    with archive.open("wb") as target:
                        received, reported = 0, -1
                        while chunk := response.read(256 * 1024):
                            target.write(chunk)
                            digest.update(chunk)
                            received += len(chunk)
                            megabytes = received // (1024 * 1024)
                            if megabytes != reported:
                                print(f"  Received {megabytes} MB", flush=True)
                                reported = megabytes
                            if time.monotonic() - start > timeout:
                                raise TimeoutError(
                                    "Lesson download exceeded its time limit."
                                )
                if digest.hexdigest() != sha256:
                    raise ValueError(
                        "Lesson download failed its SHA-256 integrity check."
                    )
                break
            except (OSError, urllib.error.URLError, TimeoutError, ValueError) as error:
                print(f"Download did not complete: {error}", flush=True)
                if attempt == attempts:
                    raise RuntimeError(
                        "Could not download verified lesson files. Check internet access, "
                        "then rerun this cell. No manual ZIP upload is required."
                    ) from error
        stage = Path(tmp) / "verified"
        stage.mkdir()
        print("Checking and unpacking the lesson archive…", flush=True)
        with zipfile.ZipFile(archive) as bundle:
            names = bundle.namelist()
            if len(names) != len(set(names)) or any(
                not (stage / name).resolve().is_relative_to(stage.resolve())
                for name in names
            ):
                raise RuntimeError(
                    "The lesson archive contains invalid or duplicate paths."
                )
            if bundle.testzip() is not None:
                raise RuntimeError("The lesson archive is damaged.")
            bundle.extractall(stage)
        if not (stage / "scripts/setup/setup.py").is_file():
            raise RuntimeError("The release is missing its setup program.")
        work.mkdir(parents=True, exist_ok=True)
        files = [name for name in names if (stage / name).is_file()]
        for name in files:
            target = work / name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(stage / name), target)
        pending = work / ".lesson_release.pending"
        pending.write_text(json.dumps({"sha256": sha256, "url": url, "files": files}))
        pending.replace(marker)
    print(f"Lesson files ready: {work}", flush=True)

install_lesson(BUNDLE_URL, BUNDLE_SHA256, WORK)
sys.path.insert(0, str(WORK / "src"))
sys.path.insert(0, str(WORK))
from bdxr_classroom.lesson.presentation import media_pair, read_json

In [ ]:
# @title Install pinned dependencies
from bdxr_classroom.lesson.process import run_logged

setup_start = time.monotonic()
run_logged(
    [sys.executable, "-u", "scripts/setup/setup.py"],
    cwd=WORK,
    log_path=WORK / "setup.log",
    timeout=1800,
    label="Installing pinned dependencies",
    show=lambda line: line.startswith(("Setup", "GPU:", "MuJoCo:", "Lesson fonts:", "Runtime details")),
)
PYTHON = str(WORK / ".venv/bin/python")
runtime_env = dict(
    os.environ,
    PYTHONPATH=str(WORK / "src"),
    MUJOCO_GL="egl",
    BDXR_RUNTIME="colab" if Path("/content").exists() else "local_notebook",
)
gpu = json.loads(
    subprocess.check_output(
        [
            PYTHON,
            "-c",
            "import torch,json; print(json.dumps(dict(name=torch.cuda.get_device_name(0),gib=torch.cuda.get_device_properties(0).total_memory/2**30)))",
        ],
        cwd=WORK,
        env=runtime_env,
        text=True,
        timeout=60,
    )
)
print("GPU:", gpu)
print("Setup time:", round(time.monotonic() - setup_start), "seconds")

In [ ]:
# @title Initialize the notebook's simulator connection
from bdxr_classroom.lesson.runtime import Lesson
from bdxr_classroom.lesson.simulator import SimulatorCells

if "simulator" in globals():
    simulator.close()
simulator = SimulatorCells(PYTHON, WORK, runtime_env)
simulator.register(get_ipython())
lab = Lesson(WORK, PYTHON, runtime_env)
(WORK / "figures").mkdir(exist_ok=True)

### Define the walking task

These are the validated baseline settings. Positive forward speed asks for
forward motion; negative speed asks for backward motion. Positive yaw rate
requests a left turn. Zero yaw requests no rotation, without a compass or
an external heading-correction controller.

`balanced` samples several command categories during training. The last two
settings reward small **sustained** turning error and sideways drift.

In [ ]:
ENV_CONFIG = {
    "task_version": "v2",  # Ten leg-joint targets; the complete robot remains physical.
    "command_mode": "balanced",  # Practice forward/backward movement, stopping, and turns.
    "speed_min": -0.25,  # m/s, backward command limit.
    "speed_max": 0.50,  # m/s, forward command limit.
    "yaw_max": 0.60,  # rad/s, magnitude of the turning command.
    "yaw_tracking": "imu",  # Estimate turning rate from gyro and gravity direction.
    "yaw_std": 0.20,  # rad/s, width of the turning reward.
    "yaw_bias_weight": 6.0,  # Weight for reducing sustained turning error.
    "lateral_bias_weight": 2.0,  # Weight for reducing sustained sideways drift.
}
(WORK / "environment_settings.json").write_text(json.dumps(ENV_CONFIG, indent=2))

Cells marked `%%robot` run visible Python in the pinned simulator environment.
Variables persist between these cells. Ordinary cells display figures and
organize experiments in the notebook.

In [ ]:
%%robot
import json
from pathlib import Path
from bdxr_classroom.simulation.environment import make_env

settings = json.loads(Path("environment_settings.json").read_text())
env = make_env(num_envs=1, seed=7, render=True, **settings)
observations, _ = env.reset(seed=7)
print("Physics timestep:", env.cfg.sim.mujoco.timestep, "seconds")
print("Policy/control timestep:", env.step_dt, "seconds")
print("Episode time limit:", env.cfg.episode_length_s, "seconds")

In [ ]:
%%robot
from bdxr_classroom.lesson.kinematics import save_pose

save_pose(env, "figures/robot.png")
print("Robot joints:", env.scene["robot"].joint_names)

In [ ]:
from IPython.display import display, Image

display(Image(filename=str(WORK / "figures/robot.png"), width=720))

The simulator advances physics every **5 ms**; the policy chooses new targets
every **20 ms**. Training episodes last up to **10 s** and also end on the
configured fall conditions. A reset starts another attempt.

**Check your understanding:** why can physics need more updates than the policy?

## 3 · Follow one leg through the kinematic tree

The floating base describes the body's position and orientation; it is not a
motor. Each leg branches through **hip yaw → hip roll → hip pitch → knee → ankle**.
The neck/head is another branch. The tree below is read from the actual model;
fixed connections are collapsed to keep it readable.

### Instructor example · joints on the actual model

| Left-leg links and joint pivots | The same chain in the model |
|:---:|:---:|
| ![Left-leg joints on BDX-R](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-annotated_robot.png?v=fa3b952929f5) | ![Articulated kinematic tree](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-kinematic_tree.png?v=318ab7e68f24) |

The left leg faces the camera and is highlighted. Colored collars mark its
measured hinge centers; these are schematic overlays, not motor CAD.
Match their numbers to the tree. Other branches are faded or collapsed.

### Your activity · inspect joints, then move one

Run the inspection to print joint axes and limits and regenerate these figures.
Then use the knee slider below to compare your prediction with forward kinematics.

In [ ]:
%%robot
from bdxr_classroom.lesson.kinematics import (
    print_joint_table,
    save_tree,
    save_annotated_pose,
)

print_joint_table(env)
save_tree(env, "figures/kinematic_tree.svg")
save_annotated_pose(env, "figures/annotated_robot.png")

In [ ]:
media_pair(
    [
        (
            "Follow the actual left-leg joints",
            WORK / "figures/annotated_robot.png",
            "Inspect the hip, knee, and ankle on one leg.",
        ),
        (
            "Actual articulated joint tree",
            WORK / "figures/kinematic_tree.svg",
            "Left and right legs have five actuated joints each. The head/neck has four.",
        ),
    ]
)

### Joint angles determine foot position

For the slider, we view the **right leg** from its visible side. Let
$\mathbf q_R$ contain its five joint angles. Forward kinematics
computes the foot position relative to the torso:

$$\mathbf p_{\mathrm{foot}} = f(\mathbf q_R), \qquad
{}^{B}T_F = {}^{B}T_1(q_1)\,{}^{1}T_2(q_2)\cdots{}^{5}T_F.$$

Each transform contributes a joint rotation and a fixed link offset. The local
rotation axes and limits are listed above. Mirrored legs can use opposite axis
signs—do not assume that equal left/right angles produce identical motion.

**Predict:** holding all other angles fixed, what path will the foot trace as
the knee turns? Is the path a straight line?

| One-joint forward kinematics |
|:---:|
| ![Measured knee-sweep foot path](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-knee_path.png?v=d1a6e64cd925) |

**Instructor example:** measured positions from −40° to +20° of right-knee motion,
with the root and all other joints fixed. The curve is geometry, not a stable gait.
Run the next cells to generate your own interactive view and change the angle range.

In [ ]:
%%robot
from bdxr_classroom.lesson.kinematics import sweep_joint

# Angles are in degrees here for readability; the model uses radians internally.
angles = range(-40, 21, 5)
foot_positions = sweep_joint(env, "Right_Knee", angles, "figures/knee_sweep.html")
print("Generated", len(foot_positions), "forward-kinematics configurations.")

In [ ]:
from bdxr_classroom.lesson.presentation import interactive_html

interactive_html(WORK / "figures/knee_sweep.html", height=610)

The slider changes one joint with the root and other joints fixed. The foot
coordinates are measured in the torso frame. This is **kinematics**, not a
demonstration that every pose is balanced. It traces one joint's motion,
not the robot's full reachable workspace.

Try a smaller angle interval or another right-leg joint within its listed limits.
Compare the geometric effect before changing any training reward.

## 4 · What does the policy observe and control?

The policy receives numbers describing the robot and the requested movement.
It outputs **ten leg-joint position targets**. Motor position controllers
generate forces to approach those targets; the policy does not directly place
the feet or teleport the robot.

$$q_i^{\mathrm{target}} = q_i^{\mathrm{default}} + s_i a_i$$

Here $a_i$ is a policy action and $s_i$ is its configured joint scale. This
connects the action vector to the joint-angle picture we just explored.

### Instructor example · the policy interface

For one robot, the baseline has **47 actor inputs**, **62 critic inputs**, and
**10 actions**. The actor uses proprioception and commands; additional simulator
measurements help the critic learn. The action scale converts a dimensionless
network output into a joint-angle offset in radians.

### Your activity · verify the interface

Print the actual observation terms and action scales below. Use the table to
explain why true base linear velocity is not available to the deployed actor.

In [ ]:
%%robot
print("Actor observation shape:", observations["actor"].shape)
print("Critic observation shape:", observations["critic"].shape)
print("Action dimension:", env.action_manager.total_action_dim)
print("Actor terms:", list(env.cfg.observations["actor"].terms))
print("Critic terms:", list(env.cfg.observations["critic"].terms))
print("Joint target scales:", env.cfg.actions["joint_pos"].scale)

| Quantity | Used by deployed actor? | Purpose |
|---|---|---|
| Angular velocity and gravity direction | Yes | Sense rotation and tilt |
| Joint positions/velocities and previous actions | Yes | Describe posture and recent motion |
| Forward-speed and yaw-rate commands | Yes | Specify the requested movement |
| True base linear velocity and contact information | No; critic/rewards may use them | Help training and evaluation |
| World position, absolute heading, terrain map | No | No hidden navigation controller |

**Actor:** chooses the next action. **Critic:** estimates how useful that action
may be during learning. Only the actor drives the deployed robot.

| Observations, actions and physics |
|:---:|
| ![Control loop: actor inputs, joint targets, physics, and rewards](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-control_loop.png?v=8462f0961d9f) |

### Define what good walking means

A reward combines several preferences:

$$r_t = w_v r_{\mathrm{speed}} + w_\omega r_{\mathrm{turn}}
      + r_{\mathrm{posture}} + r_{\mathrm{feet}} - c_{\mathrm{motion}}.$$

This is a grouped explanation; the next cell prints the actual terms and
weights. For example, an exponential tracking reward has the form
$r_{\mathrm{track}}=\exp(-e^2/\sigma^2)$: small tracking error earns more reward.
Simulator measurements may score the task without being given to the actor.

### Instructor example · selected baseline reward terms

| Term | Weight | Interpretation |
|---|---:|---|
| Forward-speed tracking | 8.0 | Follow the requested speed |
| Turn-rate tracking | 4.0 | Follow the requested yaw rate |
| Upright posture | 2.0 | Keep the body upright |
| Foot airtime | 1.5 | Encourage a useful swing interval (0.10–0.60 s) |
| Foot clearance cost | −4.0 | Penalize deviation from the 0.06 m target |
| Action-change cost | −0.1 | Discourage abrupt target changes |

These are selected terms from the actual baseline, not the complete reward.
Weights multiply different quantities with different scales; a larger weight
alone does not tell us which contribution dominates.

**Your activity:** print the full reward configuration below and identify which
term you would change to encourage quicker stepping.

In [ ]:
%%robot
from prettytable import PrettyTable

rewards = PrettyTable(["Reward term", "Weight", "Parameters"])
for name, term in env.cfg.rewards.items():
    parameters = {
        key: value
        for key, value in term.params.items()
        if isinstance(value, (int, float, str, bool, tuple))
    }
    rewards.add_row([name, term.weight, str(parameters)])
print(rewards)

Speed/yaw rewards ask for command tracking. Posture and foot terms encourage
usable movement. Motion penalties discourage excessive effort or abrupt changes.
Later we will change one interpretable reward setting and inspect the result.
A higher return after changing weights does **not** automatically mean a better gait.

**Discuss:** why could rewarding only forward speed produce a poor walker?

## 5 · Configure Soft Actor-Critic

**SAC** learns from experience saved in a replay buffer. The actor proposes
actions; two critics estimate future reward. Training uses the smaller critic
estimate to reduce overly optimistic targets. An entropy term encourages
exploration rather than choosing the same action every time.

$$J(\pi)=\mathbb{E}\left[\sum_t\gamma^t
\left(r_t+\alpha\,\mathcal H(\pi(\cdot\mid o_t))\right)\right]$$

Read it as **future rewards + encouragement to explore**. $\gamma$ discounts
distant rewards; $\alpha$ controls the entropy contribution. FastSAC makes this
off-policy learning loop efficient on the GPU. This lesson uses SAC, not PPO.

| Learning from replay |
|:---:|
| ![SAC: collect experience, sample replay, update actor and critics](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-sac_loop.png?v=80ea8eb4aaa7) |

### Separate learning settings from environment settings

`num_envs` counts independent robots collecting experience. `batch_size` counts
replay samples in a neural-network update. These are different quantities.
`steps` counts parallel simulation steps: with 512 robots, one step collects
512 transitions. The training curve plots collected transitions.

| Learning setting | Recorded baseline | What it changes |
|---|---|---|
| Parallel robots | 512 | Experience collected each simulation step |
| Batch size | 2,048 | Replay samples used per update |
| Simulation steps | 10,000 | Up to 5.12 million collected transitions |
| Discount | 0.97 | Emphasis on future reward |
| Updates per step | 4 | Learning work per collection step |

### Your activity · configure and inspect SAC

Keep these settings for the first run. Inspect the network and store one
transition before launching training. A single untrained action is not walking.
A matching seed does not guarantee identical GPU training. Use the command
checks below to judge your result, rather than expecting the same video.

In [ ]:
LEARNING_CONFIG = {
    "seed": 7,  # Same initialization seed as the recorded baseline.
    "num_envs": 512,  # Independent simulations collecting experience.
    "batch_size": 2048,  # Replay samples per learning update.
    "buffer_size": 512,  # Replay slots per parallel robot.
    "steps": 10000,  # Parallel simulation steps.
    "minutes": 15,  # Stop earlier if the training-time limit is reached.
    "gamma": 0.97,  # Discount applied to future reward.
    "updates": 4,  # Learning updates per simulation step.
    "actor_hidden": 256,  # Actor hidden-layer width.
    "critic_hidden": 384,  # Critic hidden-layer width.
    "mirror_replay": True,  # Reuse left/right mirrored experience.
    "compile": True,  # Compile the repeated GPU learning operations.
}
USE_WANDB = False  # @param {type:"boolean"}
LEARNING_CONFIG["wandb_mode"] = "online" if USE_WANDB else "disabled"

In [ ]:
# @title Optional W&B connection
if USE_WANDB and not (
    os.environ.get("WANDB_API_KEY") or Path.home().joinpath(".netrc").exists()
):
    import getpass

    lab.environment["WANDB_API_KEY"] = getpass.getpass(
        "W&B API key (not stored in the lesson): "
    )
print("W&B:", LEARNING_CONFIG["wandb_mode"])
print("Local metrics and checkpoints are saved even when W&B is disabled.")

The recorded baseline took about **9 minutes** of training on an L40S GPU.
Setup, compilation, evaluation, and video rendering take additional time.
A Colab GPU or a different seed may take longer or fail the quality checks.

**Predict:** what changes if we double the number of parallel environments but
keep the number of simulation steps unchanged?

### Inspect the network and one interaction

Before the long training call, create a small inspection instance with the same
network widths. It uses one robot, eight replay slots, and no compilation or
mirroring solely to make the data flow easy to inspect. The full training
configuration above stays unchanged.

In [ ]:
(WORK / "learning_settings.json").write_text(json.dumps(LEARNING_CONFIG, indent=2))

In [ ]:
%%robot
from bdxr_classroom.learning.fastsac import FastSAC

inspection = json.loads(Path("learning_settings.json").read_text())
inspection.update(num_envs=1, buffer_size=8, compile=False, mirror_replay=False)
agent = FastSAC(
    observations["actor"].shape[-1],
    observations["critic"].shape[-1],
    env.action_manager.total_action_dim,
    inspection,
)
print(agent.actor)

In [ ]:
%%robot
# An untrained actor chooses an action; physics determines the result.
actions = agent.act(observations["actor"])
next_observations, reward, terminated, truncated, info = env.step(actions)
agent.add(observations, actions, next_observations, reward, terminated, truncated)
print("Action shape:", actions.shape)
print("Reward:", reward)
print("Replay write index after storing this transition:", agent.buffer.ptr)

This is one **collection step**, not a trained policy. The full training program
repeats collection, samples replay batches, and updates the critics and actor.
`terminated` marks a task termination; `truncated` marks a time limit. Keeping
the final observation lets the learner handle those endings correctly.

In [ ]:
%%robot
# Release the inspection environment before the full training jobs.
env.close()

In [ ]:
simulator.close()

## 6 · Train your walking controller

This call starts a fresh actor and critics using the two configurations above.
Checkpoints, exact settings, timings, and real learning metrics are saved in
a timestamped experiment folder. The helper only runs the training program
and manages its output paths; the learning settings remain visible here.

### Instructor example · a real fresh-training run

| Learning: episode return | Independent check: lateral drift |
|:---:|:---:|
| ![Fresh seed-7 walking learning curve](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-scratch_learning.png?v=596a6bd14a3d) | ![Measured lateral drift during forward walking](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-scratch_forward.png?v=178e45093c1a) |

Seed 7, trained from scratch. Rising return suggests learning; the drift plot
checks straightness. Each line is one trial. World position is evaluation-only.

### Your activity · train, record, and evaluate

Train, record your before/after videos, and check command following. Compare
your results with the examples above.

In [ ]:
BASELINE = lab.train("command_walking_baseline", ENV_CONFIG, LEARNING_CONFIG)
INITIAL_BASELINE = BASELINE
print("Saved experiment:", BASELINE)

In [ ]:
# Both recordings use checkpoints from this training run.
before = lab.record(BASELINE, checkpoint="initial")
after = lab.record(BASELINE, checkpoint="final")
media_pair(
    [
        ("Before learning", before, "Initial randomly initialized actor."),
        (
            "After the first training attempt",
            after,
            "This attempt still needs the command checks below.",
        ),
    ]
)

In [ ]:
from IPython.display import display, Image

display(Image(filename=str(lab.learning_curve(BASELINE)), width=720))
print("Actual timing:", read_json(BASELINE / "metrics/timing.json"))

### Does it do what we asked?

Evaluate slow/normal/fast forward walking, backward walking, stopping, turns,
and a command sequence. Use five perturbed starting states for each command.
These are **evaluation trials of one trained policy**, not five training seeds.

The fixed acceptance limits check speed error, yaw-rate error, straight-line
drift, complete 15-second trials, and zero falls. Curved paths are expected
when the command requests a turn. World position is measured for evaluation;
it is not an actor input.
These checks can take several minutes; progress identifies each command.

In [ ]:
from bdxr_classroom.lesson.assessment import (
    check_recent_walking,
    evaluate_walking,
    selected_walking,
)
from bdxr_classroom.lesson.presentation import command_table

BASELINE_SUMMARY, BASELINE_CHECK = evaluate_walking(lab, BASELINE)
command_table(BASELINE_SUMMARY)
print("Passed:", BASELINE_CHECK["passed"])
print("Failures:", BASELINE_CHECK["failures"])

A later update can make a policy worse. If the final checkpoint fails, check
up to two recent saved snapshots using the **same full command suite**. This
adds evaluation time, not training. Each result stays in its own folder.

If none passes, allow one additional training stage
with the **same behavior settings**. This fine-tunes saved networks using fresh
replay; it is not an exact replay-buffer resume. Preserve and inspect both runs.
A recovered policy has its own folder and video; it does not replace the first attempt.

In [ ]:
BASELINE_SUMMARY, BASELINE_CHECK = check_recent_walking(
    lab, BASELINE, BASELINE_SUMMARY, BASELINE_CHECK
)
if not BASELINE_CHECK["passed"]:
    recovery = {**LEARNING_CONFIG, "steps": 3000, "minutes": 10}
    BASELINE = lab.train(
        "command_walking_recovery",
        ENV_CONFIG,
        recovery,
        parent=BASELINE / "checkpoints/final.pt",
    )
    BASELINE_SUMMARY, BASELINE_CHECK = evaluate_walking(lab, BASELINE)
    BASELINE_SUMMARY, BASELINE_CHECK = check_recent_walking(
        lab, BASELINE, BASELINE_SUMMARY, BASELINE_CHECK
    )
    command_table(BASELINE_SUMMARY)
SELECTED_WALK_CHECKPOINT = selected_walking([BASELINE_CHECK])
if not BASELINE_CHECK["passed"]:
    raise RuntimeError(
        "No checked snapshot passed. Keep the reports and inspect the failed "
        "commands before trying another seed or a longer training budget."
    )
print("Selected walking checkpoint:", SELECTED_WALK_CHECKPOINT)
print("Passed command checks:", BASELINE_CHECK["passed"])

In [ ]:
from IPython.display import display, Image, Video

# Keep the robot video separate from the smaller diagnostic chart.
selected_evaluation = Path(BASELINE_CHECK["evaluation_directory"])
selected_video = BASELINE / "videos" / selected_evaluation.name / "forward.mp4"
print("Selected-policy evaluation:", selected_evaluation)
display(Video(str(selected_video), embed=True, width=640))
display(Image(filename=str(selected_evaluation / "forward.png"), width=720))

**Interpret:** can you point to evidence for straight walking, beyond the video?
If reward rises while the controller still falls, is training finished?

## 7 · Change one thing and train again

Commanding a trained robot to walk faster is different from **changing a reward
and training a different gait**. Here we hold the command range fixed and alter
one interpretable behavior setting. Train the variant and an unchanged control
from scratch with the same seed and step budget.

| Experiment | Changed reward setting | What to inspect |
|---|---|---|
| Level feet | Sole-level reward weight 0.75 | Foot tilt |
| Quick steps | Airtime window 0.06–0.16 s, weight 4 | Step cadence |
| Crouched walking | Torso-height target 0.23 m, fixed weight 4 | Body height and foot clearance |

Quick steps changes one reward's window and weight together; it is not a
one-scalar ablation. All three recorded variants passed the full command suite,
with one successful training seed per style. That is an example, not proof that
every seed will succeed.

### A recorded example to compare with your prediction

This instructor comparison is shown at equal commands. Local variant training
took 14m54s (level feet), 15m1s (quick steps), or 11m13s (crouched).
The unchanged seed-8 control missed command checks; it is labeled as a matched
comparison, not another validated baseline. The selected crouched checkpoint
is the 9,000-step result; training longer did not improve its acceptance checks.

#### Instructor example · Flatter feet

| Matched-command example | Measured gait differences |
|:---:|:---:|
| ![Recorded flatter-foot comparison at matched commands](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/inline/level_feet.gif?v=8e8e9452446e) | ![Measured gait differences](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-level_feet_focus.png?v=ce6d0d4f1290) |

#### Instructor example · Quicker steps

| Matched-command example | Measured gait differences |
|:---:|:---:|
| ![Recorded quick-step comparison at matched commands](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/inline/quick_steps.gif?v=c3e2fc28dbd2) | ![Measured gait differences](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-quick_steps_focus.png?v=93d6218c1115) |

#### Instructor example · Crouched walking

| Matched-command example | Measured gait differences |
|:---:|:---:|
| ![Recorded crouched comparison at matched commands](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/inline/crouched.gif?v=873f03b8ce02) | ![Measured gait differences](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/reference-crouched_focus.png?v=dfd330039b2b) |

### Your turn

Predict a measurable change. Train a matched control and variant, then compare
your videos, measurements, and command checks.

In [ ]:
EXPERIMENT = "level_feet"  # @param ["level_feet", "quick_steps", "crouched"]
PREDICTION = "The feet will stay flatter during stepping."  # @param {type:"string"}
CHANGES = {
    "level_feet": {"level_feet_weight": 0.75},
    "quick_steps": {"air_time_min": 0.06, "air_time_max": 0.16, "air_time_weight": 4.0},
    "crouched": {"walking_height": 0.23},
}
VARIANT_ENV = {**ENV_CONFIG, **CHANGES[EXPERIMENT]}
PAIR_LEARNING = {
    **LEARNING_CONFIG,
    "seed": 8 if EXPERIMENT == "crouched" else 7,
    "steps": 9000 if EXPERIMENT == "crouched" else 10000,
}
print("Prediction:", PREDICTION)
print("Only changed behavior setting:", CHANGES[EXPERIMENT])

In [ ]:
# Neither call has a parent checkpoint: both start from scratch.
CONTROL = lab.train("scratch_control", ENV_CONFIG, PAIR_LEARNING)
VARIATION = lab.train(EXPERIMENT, VARIANT_ENV, PAIR_LEARNING)

In [ ]:
CONTROL_SUMMARY, CONTROL_CHECK = evaluate_walking(lab, CONTROL)
VARIATION_SUMMARY, VARIATION_CHECK = evaluate_walking(lab, VARIATION)
command_table(CONTROL_SUMMARY)
command_table(VARIATION_SUMMARY)
print("Control passes:", CONTROL_CHECK["passed"])
print("Variation passes:", VARIATION_CHECK["passed"])

In [ ]:
# Same commanded speed makes the visual comparison meaningful.
control_video = lab.record(CONTROL, speed=0.30)
variation_video = lab.record(VARIATION, speed=0.30)
media_pair(
    [
        (
            "Your unchanged control",
            control_video,
            "Fresh training at the matched seed and budget.",
        ),
        ("Your variation", variation_video, f"Changed setting: {CHANGES[EXPERIMENT]}"),
    ]
)

In [ ]:
from bdxr_classroom.lesson.gait import compare_gaits

compare_gaits(lab, CONTROL, VARIATION)
media_pair(
    [
        (
            "Control learning curve",
            lab.learning_curve(CONTROL),
            "Read the trend; do not compare reward magnitude alone.",
        ),
        (
            "Variation learning curve",
            lab.learning_curve(VARIATION),
            "The reward definition changed between experiments.",
        ),
    ]
)
SELECTED_WALK_CHECKPOINT = selected_walking(
    [BASELINE_CHECK, CONTROL_CHECK, VARIATION_CHECK]
)

**Explain your result:** did the gait change as predicted, and did command
following remain good? A different-looking policy that falls is not a successful
variant. Check actual step counts: a time limit can stop one run earlier and
spoil the equal-budget comparison. Preserve failures in your report.

## 8 · Optional extension: learn to hop

A hop lifts **both feet** before a controlled landing. This recorded hopper
reached about 5.9 cm bilateral clearance and 0.16 s of clear flight; its feet
still tilt about 9°. Foot-marker height alone does not prove obstacle clearance.

| Earlier vs updated hopping | Same motion at half speed |
|:---:|:---:|
| ![Recorded hopping comparison](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/inline/hopping.gif?v=a07c95656f35) | ![Recorded hopping comparison at half speed](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/inline/hopping_slow.gif?v=4c47173d257a) |

Watch both feet leave the ground, then inspect the landing.

### A curriculum adds one skill at a time

**Rhythm → clearance → foot control → landing.** The goal is not to make every
row taller. Some stages return to flat ground to isolate a new skill before
adding obstacles again. Each stage starts from its predecessor's networks.

| Phase | Stages, in order | What changes—and why |
|---|---|---|
| **1 · Find a rhythm** | 1. Tiny hops → 2. Moving hops | Start on flat ground; then add speed and turning commands. |
| **2 · Clear the ground** | 3. Terrain hops | Measure the whole sole in flight and introduce 25 mm terrain. |
| **3 · Control both feet** | 4. Level hops → 5. Level terrain hops | Reward simultaneous clearance and flatter soles on flat ground; then restore terrain. |
| **4 · Refine the landing** | 6. Natural hops → 7. Block hops → 8. Soft landing hops | Level feet during flight only, practise steps, then reduce body-lift reward. |

The last phases keep the same 6.5 cm foot target while changing **how** the
robot takes off, clears terrain, and lands. The eight recorded stages took
about **23 minutes** on an L40S, excluding setup and evaluation.

| Recorded hopping measurements |
|:---:|
| ![Measured instructor hopping](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/course_hops-motion.png?v=3a4a6eee5854) |

### Your optional activity · train and validate hopping

Enable the toggle below to generate your own policies and videos. Leaving it
off does not train or download an instructor checkpoint.

In [ ]:
from bdxr_classroom.lesson.hopping import load_curriculum, train_stage, validate_pair

CURRICULUM = load_curriculum(WORK)

In [ ]:
from IPython.display import display, Video

TRAIN_HOPPING = False  # @param {type:"boolean"}
HOP_STAGES = []
SELECTED_HOP_CHECKPOINT = None
if TRAIN_HOPPING:
    previous = None
    for stage in CURRICULUM:
        print("Training stage:", stage["name"])
        previous = train_stage(lab, stage, LEARNING_CONFIG, previous)
        HOP_STAGES.append(previous)
        display(Video(str(lab.record(previous)), embed=True, width=640))

### Select evidence, not just the latest checkpoint

Saved earlier checkpoints can outperform the final checkpoint. Screen the exact
weights on terrain, then test repeated walking/hopping handovers on flat ground
and a step. Every recorded failure stays in the history; a later success does
not silently erase it. Passing these checks does not establish full-course skill.

In [ ]:
from IPython.display import display, Image

if HOP_STAGES:
    SELECTED_HOP_CHECKPOINT = validate_pair(
        lab, HOP_STAGES[-1], SELECTED_WALK_CHECKPOINT
    )
    display(Image(filename=str(lab.learning_curve(HOP_STAGES[-1])), width=900))
    print("Selected hopping checkpoint:", SELECTED_HOP_CHECKPOINT)
else:
    print(
        "Hopping training skipped. An existing compatible checkpoint can be loaded in the game."
    )

**Discuss:** why might two good policies still fail when switching between them?
Consider command speed, phase, and abrupt changes in joint targets.

## 9 · Play: Desert Droid Run

Drive through **three obstacles**, cross the rough patch, and reach the raised
finish beacon. You provide speed, turn-rate, and mode commands; the trained
controller supplies leg actions. There is no navigation planner.

The middle step is 25 mm high. The selected hopper crossed 3/3 tested starts;
the walking controller crossed 0/6 at normal/fast commands. This supports hopping
for these controllers and trials, not a claim that every walker must fail.
The refreshed visually steered run finished without falls in 53.56 simulated
seconds. Fixed command replays can miss the finish; players need to steer.

### Instructor example · the course

| The course |
|:---:|
| ![Desert Droid Run terrain](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/course_preview.png?v=c3bf346cee5c) |

Locate the three obstacles, rough ground, and raised finish before watching the run.
You will steer your own trained controller through this same world.

| Why hopping matters | Course preview |
|:---:|:---:|
| ![Walking versus hopping over the obstacle](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/inline/obstacle.gif?v=67331bb28a8d) | ![First eight seconds of the recorded instructor run](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/inline/course.gif?v=5d70c51754d2) |

Left: the same step at two walking speeds and with hopping. Right: a visually
steered run’s first eight seconds. These are recorded examples.

### Your live interactive game

Run the next cell to open the live control panel here. Keep the runtime connected.
Leave paths blank to use your selected policies, or enter compatible checkpoints.

**Click inside:** W/S moves, A/D turns, Space switches walk/hop. Release keys to
stop, or use the on-screen buttons. Hopping needs a trained hopping checkpoint.
Switch before the obstacle and allow the robot to settle after landing.

**Fall → reset; time accumulates.** Reach the raised finish upright to win.
On remote VS Code, forward port 8765 if the panel is blank. No plugin is needed.


In [ ]:
# @title Start the interactive game with your trained controller
from bdxr_classroom.lesson.play import launch_game, display_game, pause_and_recordings

WALK_CHECKPOINT = ""  # @param {type:"string"}
HOP_CHECKPOINT = ""  # @param {type:"string"}
GAME_PORT = 8765
walking = WALK_CHECKPOINT or SELECTED_WALK_CHECKPOINT
hopping = HOP_CHECKPOINT or SELECTED_HOP_CHECKPOINT
status = launch_game(lab, walking, hopping, port=GAME_PORT)
display_game(GAME_PORT)
print("Game recordings and score:", status["experiment"])

## 10 · Save and explain your results

Pause the game before exporting. The backup includes your training stages,
configurations, checkpoints, curves, and recorded gameplay. Including replay
makes the file larger but preserves collected experience for continuation.
Setup logs, package versions, and the lesson release are included for troubleshooting.

### What your evidence should contain

| Evidence | Question it answers |
|---|---|
| Config, seed, actual steps and timing | What exactly did you train? |
| Checkpoint and before/after video | What behavior did learning produce? |
| Learning curve and command checks | Did reward improve, and does it work? |
| Matched gait comparison | Did the chosen reward change movement? |
| Gameplay recording and score | Where did your controller succeed or fall? |

### Your activity · export your results

The cells below show your gameplay recordings and package your runs. Instructor
examples remain in the lesson; they are not evidence that your training passed.

In [ ]:
from IPython.display import display, Video

recordings = pause_and_recordings(lab, GAME_PORT)
for video in recordings:
    display(Video(str(video), embed=True, width=720))
if not recordings:
    print("No gameplay video yet. Drive the droid, then rerun this cell.")

In [ ]:
from bdxr_classroom.lesson.exports import export_results

INCLUDE_REPLAY = True  # @param {type:"boolean"}
backup = export_results(WORK, lab.runs, include_replay=INCLUDE_REPLAY)
print("Saved:", backup)
try:
    from google.colab import files

    files.download(str(backup))
except ImportError:
    print("Retrieve this ZIP from the local lesson folder.")

To continue later, unpack the backup into the lesson folder. The training CLI's
`--resume <run>/checkpoints/resume.pt` restores saved networks, optimizers,
normalization, and replay; simulated episodes restart. Branching from an earlier
`step_*.pt` with `--finetune` starts new replay and is a different operation.

**Your short lab report:**

1. Explain how one joint angle changes foot position using the kinematic study.
2. Identify the actor's observations and explain why it cannot use world heading.
3. Show your baseline video and evidence for command tracking and straightness.
4. State your changed reward, prediction, seed, actual training budget, and result.
5. Explain one failure or limitation—including whether both runs passed the checks.

Kinematics describes the motion geometry. Dynamics determines whether forces
and contacts can realize it. Reinforcement learning searches for actions that
earn reward; videos and independent measurements tell us what it actually learned.